In [58]:
import pandas as pd

In [59]:
files = [
    "Cherkasy.csv", "Chernihiv.csv", "Chernivtsi.csv", "Dnipro.csv", "Ivano-Frankivsk.csv",
    "Kharkiv.csv", "Kherson.csv", "Khmelnytskyi.csv", "Kropyvnytskyi.csv",
    "Lutsk.csv", "Lviv.csv", "Mykolaiv.csv", "Odesa.csv", "Poltava.csv", "Rivne.csv",
    "Sumy.csv", "Ternopil.csv", "Uzhhorod.csv", "Vinnytsia.csv", "Zaporizhzhia.csv",
    "Zhytomyr.csv"
]

dfs = []
for filename in files:
    df = pd.read_csv(filename)
    dfs.append(df)
    print(f"✓ {filename}")

df_ukraine = pd.concat(dfs, ignore_index=True)

✓ Cherkasy.csv
✓ Chernihiv.csv
✓ Chernivtsi.csv
✓ Dnipro.csv
✓ Ivano-Frankivsk.csv
✓ Kharkiv.csv
✓ Kherson.csv
✓ Khmelnytskyi.csv
✓ Kropyvnytskyi.csv
✓ Lutsk.csv
✓ Lviv.csv
✓ Mykolaiv.csv
✓ Odesa.csv
✓ Poltava.csv
✓ Rivne.csv
✓ Sumy.csv
✓ Ternopil.csv
✓ Uzhhorod.csv
✓ Vinnytsia.csv
✓ Zaporizhzhia.csv
✓ Zhytomyr.csv


In [60]:
df_ukraine.info()

<class 'pandas.DataFrame'>
RangeIndex: 12464 entries, 0 to 12463
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   num_of_rooms       12464 non-null  int64  
 1   freshly_renovated  12384 non-null  object 
 2   area               12393 non-null  float64
 3   living_area        8616 non-null   float64
 4   kitchen_area       12373 non-null  float64
 5   floor              12455 non-null  float64
 6   floors_in_house    12455 non-null  float64
 7   year_of_building   8033 non-null   float64
 8   price              12436 non-null  float64
 9   house_type         12464 non-null  str    
 10  heating            12464 non-null  str    
 11  wall_type          12464 non-null  str    
 12  url                12464 non-null  str    
 13  district           12464 non-null  str    
 14  city               12464 non-null  str    
 15  geo_region         12464 non-null  str    
dtypes: float64(7), int64(1), object(1

In [61]:
df_kyiv = pd.read_csv("Kyiv.csv")

In [62]:
df_kyiv.info()

<class 'pandas.DataFrame'>
RangeIndex: 10958 entries, 0 to 10957
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   num_of_rooms       10957 non-null  float64
 1   freshly_renovated  10823 non-null  object 
 2   area               10931 non-null  float64
 3   living_area        6183 non-null   float64
 4   kitchen_area       10920 non-null  float64
 5   floor              10954 non-null  float64
 6   floors_in_house    10954 non-null  float64
 7   year_of_building   10804 non-null  float64
 8   price              10953 non-null  float64
 9   house_type         10958 non-null  str    
 10  heating            10958 non-null  str    
 11  wall_type          10958 non-null  str    
 12  url                10958 non-null  str    
 13  district           10958 non-null  str    
 14  city               10958 non-null  str    
 15  geo_region         10958 non-null  str    
dtypes: float64(8), object(1), str(7)


In [63]:
from sklearn.impute import KNNImputer

NUMERIC_TARGETS = [
    "num_of_rooms", "area", "living_area", "kitchen_area",
    "floor", "floors_in_house", "year_of_building", "price",
]

ROUND_TO_INT = ["num_of_rooms", "floor", "floors_in_house", "year_of_building"]

CATEGORICAL_COMPLETE = ["house_type", "heating", "wall_type"]

PASSTHROUGH_UNTOUCHED = [
    "url",
    "is_low_floor", "is_high_floor", "is_middle_floor",
    "is_large_city", "is_close_to_eu", "is_near_border_threat", "is_tourist_hub",
]


def knn_impute_apartments(
    df: pd.DataFrame,
    location_cols: list[str],
    n_neighbors: int = 5,
) -> pd.DataFrame:
    df = df.copy()

    passthrough_cols = [c for c in PASSTHROUGH_UNTOUCHED if c in df.columns]
    passthrough = df[passthrough_cols].copy()

    fr = df["freshly_renovated"].map({True: 1.0, False: 0.0})

    cat_cols = [c for c in CATEGORICAL_COMPLETE if c in df.columns] + location_cols
    onehot = pd.get_dummies(df[cat_cols], columns=cat_cols, dummy_na=False)

    numeric_targets = [c for c in NUMERIC_TARGETS if c in df.columns]

    feature_df = pd.concat(
        [df[numeric_targets], fr.rename("freshly_renovated"), onehot],
        axis=1,
    )

    scaler = StandardScaler()
    scaled = scaler.fit_transform(feature_df)

    imputer = KNNImputer(n_neighbors=n_neighbors, weights="distance")
    imputed_scaled = imputer.fit_transform(scaled)

    imputed = pd.DataFrame(
        scaler.inverse_transform(imputed_scaled),
        columns=feature_df.columns,
        index=feature_df.index,
    )

    result = df.copy()
    for col in numeric_targets:
        result[col] = imputed[col]
    for col in ROUND_TO_INT:
        if col in result.columns:
            result[col] = result[col].round().astype("Int64")

    result["freshly_renovated"] = imputed["freshly_renovated"].round().astype(int).astype(bool)

    for col in passthrough_cols:
        result[col] = passthrough[col]

    return result


if __name__ == "__main__":

    df_kyiv = knn_impute_apartments(df_kyiv, location_cols=["district"])
    df_ukraine = knn_impute_apartments(df_ukraine, location_cols=["district", "city"])

In [64]:
df_ukraine.info()

<class 'pandas.DataFrame'>
RangeIndex: 12464 entries, 0 to 12463
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   num_of_rooms       12464 non-null  Int64  
 1   freshly_renovated  12464 non-null  bool   
 2   area               12464 non-null  float64
 3   living_area        12464 non-null  float64
 4   kitchen_area       12464 non-null  float64
 5   floor              12464 non-null  Int64  
 6   floors_in_house    12464 non-null  Int64  
 7   year_of_building   12464 non-null  Int64  
 8   price              12464 non-null  float64
 9   house_type         12464 non-null  str    
 10  heating            12464 non-null  str    
 11  wall_type          12464 non-null  str    
 12  url                12464 non-null  str    
 13  district           12464 non-null  str    
 14  city               12464 non-null  str    
 15  geo_region         12464 non-null  str    
dtypes: Int64(4), bool(1), float64(4),

In [65]:
df_ukraine['living_ratio'] = df_ukraine['living_area'] / df_ukraine['area']

large_cities = ['Lviv', 'Odesa', 'Dnipro', 'Kharkiv']
df_ukraine['is_large_city'] = df_ukraine['city'].isin(large_cities).astype(int)

border_eu_cities = ['Lviv', 'Uzhhorod', 'Ivano-Frankivsk', 'Lutsk', 'Chernivtsi', 'Ternopil', 'Rivne']
df_ukraine['is_close_to_eu'] = df_ukraine['city'].isin(border_eu_cities).astype(int)

border_risk_cities = ['Sumy', 'Chernihiv', 'Kharkiv', 'Zhytomyr', 'Lutsk', 'Rivne']
df_ukraine['is_near_border_threat'] = df_ukraine['city'].isin(border_risk_cities).astype(int)

tourist_cities = ['Lviv', 'Odesa', 'Ivano-Frankivsk', 'Uzhhorod', 'Chernivtsi']
df_ukraine['is_tourist_hub'] = df_ukraine['city'].isin(tourist_cities).astype(int)

df_ukraine['is_low_floor'] = 0
df_ukraine['is_high_floor'] = 0

high_mask = df_ukraine['floors_in_house'] > 6
df_ukraine.loc[high_mask & (df_ukraine['floor'] <= 3), 'is_low_floor'] = 1
df_ukraine.loc[high_mask & (df_ukraine['floor'] > (df_ukraine['floors_in_house'] - 3)), 'is_high_floor'] = 1

low_mask = df_ukraine['floors_in_house'] <= 6
df_ukraine.loc[low_mask & (df_ukraine['floor'] == 1), 'is_low_floor'] = 1
df_ukraine.loc[low_mask & (df_ukraine['floor'] == df_ukraine['floors_in_house']), 'is_high_floor'] = 1

In [66]:
df_kyiv['living_ratio'] = df_kyiv['living_area'] / df_kyiv['area']

df_kyiv['is_low_floor'] = 0
df_kyiv['is_high_floor'] = 0
df_kyiv['is_middle_floor'] = 0

high_building_mask = df_kyiv['floors_in_house'] > 6

df_kyiv.loc[high_building_mask & (df_kyiv['floor'] <= 3), 'is_low_floor'] = 1
df_kyiv.loc[high_building_mask & (df_kyiv['floor'] > (df_kyiv['floors_in_house'] - 3)), 'is_high_floor'] = 1
df_kyiv.loc[high_building_mask & (df_kyiv['floor'] > 3) & (df_kyiv['floor'] <= (df_kyiv['floors_in_house'] - 3)), 'is_middle_floor'] = 1

low_building_mask = df_kyiv['floors_in_house'] <= 6

df_kyiv.loc[low_building_mask & (df_kyiv['floor'] == 1), 'is_low_floor'] = 1
df_kyiv.loc[low_building_mask & (df_kyiv['floor'] == df_kyiv['floors_in_house']), 'is_high_floor'] = 1
df_kyiv.loc[low_building_mask & (df_kyiv['floor'] > 1) & (df_kyiv['floor'] < df_kyiv['floors_in_house']), 'is_middle_floor'] = 1

In [67]:
df_ukraine.drop_duplicates(ignore_index=True, inplace=True)
df_kyiv.drop_duplicates(ignore_index=True, inplace=True)

In [68]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler


def find_best_k(df: pd.DataFrame, col: str = "year_of_building", k_range=range(2, 9)) -> None:
    X = df[[col]].values.astype(float)
    Xs = StandardScaler().fit_transform(X)
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(Xs)
        sil = silhouette_score(Xs, km.labels_)
        print(f"k={k}: silhouette={sil:.3f}, inertia={km.inertia_:.1f}")


def cluster_by_year(df: pd.DataFrame, n_clusters: int = 3, col: str = "year_of_building") -> pd.DataFrame:
    df = df.copy()
    X = df[[col]].values.astype(float)
    Xs = StandardScaler().fit_transform(X)
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10).fit(Xs)
    df["year_cluster"] = km.labels_
    return df


def summarize_clusters(df: pd.DataFrame, col: str = "year_of_building") -> pd.DataFrame:
    return (
        df.groupby("year_cluster")
        .agg(
            n=(col, "size"),
            year_min=(col, "min"),
            year_max=(col, "max"),
            year_mean=(col, "mean"),
            price_mean=("price", "mean"),
            area_mean=("area", "mean"),
        )
        .sort_values("year_min")
    )


if __name__ == "__main__":
    for df, name in [(df_ukraine, "Ukraine"), (df_kyiv, "Kyiv")]:
        df = df

        print(f"=== {name}: підбір k ===")
        find_best_k(df)

        df_clustered = cluster_by_year(df, n_clusters=3)
        print(f"\n=== {name}: підсумок по кластерах (k=3) ===")
        print(summarize_clusters(df_clustered))
        print()

=== Ukraine: підбір k ===
k=2: silhouette=0.690, inertia=4832.5
k=3: silhouette=0.750, inertia=2032.9
k=4: silhouette=0.751, inertia=1318.0
k=5: silhouette=0.694, inertia=870.5
k=6: silhouette=0.697, inertia=667.2
k=7: silhouette=0.610, inertia=505.2
k=8: silhouette=0.583, inertia=400.7

=== Ukraine: підсумок по кластерах (k=3) ===
                 n  year_min  year_max    year_mean   price_mean  area_mean
year_cluster                                                               
2              583      1400      1937  1899.487136  1622.917667  75.538103
0             5005      1938      1996  1975.083516   824.518496  52.361676
1             6876      1997      2028  2018.124346  1417.676951  72.482196

=== Kyiv: підбір k ===
k=2: silhouette=0.734, inertia=2856.5
k=3: silhouette=0.756, inertia=1115.9
k=4: silhouette=0.648, inertia=684.6
k=5: silhouette=0.621, inertia=423.1
k=6: silhouette=0.609, inertia=329.2
k=7: silhouette=0.612, inertia=253.9
k=8: silhouette=0.582, inertia=193.3



In [69]:
df_ukraine["is_historic"] = (df_ukraine["year_of_building"] <= 1937).astype(int)
df_ukraine["is_soviet"] = ((df_ukraine["year_of_building"] >= 1938) & (df_ukraine["year_of_building"] <= 1996)).astype(int)
df_ukraine["is_modern"] = (df_ukraine["year_of_building"] >= 1997).astype(int)

In [70]:
df_kyiv["is_historic"] = (df_kyiv["year_of_building"] <= 1943).astype(int)
df_kyiv["is_soviet"] = ((df_kyiv["year_of_building"] >= 1944) & (df_kyiv["year_of_building"] <= 1994)).astype(int)
df_kyiv["is_modern"] = (df_kyiv["year_of_building"] >= 1995).astype(int)

In [71]:
df_ukraine.to_csv("Ukraine_for_analysis.csv", index=False)
df_kyiv.to_csv("Kyiv_for_analysis.csv", index=False)

In [72]:
df_ukraine.info()

<class 'pandas.DataFrame'>
RangeIndex: 12464 entries, 0 to 12463
Data columns (total 26 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   num_of_rooms           12464 non-null  Int64  
 1   freshly_renovated      12464 non-null  bool   
 2   area                   12464 non-null  float64
 3   living_area            12464 non-null  float64
 4   kitchen_area           12464 non-null  float64
 5   floor                  12464 non-null  Int64  
 6   floors_in_house        12464 non-null  Int64  
 7   year_of_building       12464 non-null  Int64  
 8   price                  12464 non-null  float64
 9   house_type             12464 non-null  str    
 10  heating                12464 non-null  str    
 11  wall_type              12464 non-null  str    
 12  url                    12464 non-null  str    
 13  district               12464 non-null  str    
 14  city                   12464 non-null  str    
 15  geo_region   

In [74]:
df_ukraine.drop(["url"], axis=1, inplace=True)
df_kyiv.drop(["url"], axis=1, inplace=True)

In [75]:
df_ukraine = df_ukraine[df_ukraine["price"].notnull()]
df_kyiv = df_kyiv[df_kyiv["price"].notnull()]

In [76]:
df_ukraine.info()

<class 'pandas.DataFrame'>
RangeIndex: 12464 entries, 0 to 12463
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   num_of_rooms           12464 non-null  Int64  
 1   freshly_renovated      12464 non-null  bool   
 2   area                   12464 non-null  float64
 3   living_area            12464 non-null  float64
 4   kitchen_area           12464 non-null  float64
 5   floor                  12464 non-null  Int64  
 6   floors_in_house        12464 non-null  Int64  
 7   year_of_building       12464 non-null  Int64  
 8   price                  12464 non-null  float64
 9   house_type             12464 non-null  str    
 10  heating                12464 non-null  str    
 11  wall_type              12464 non-null  str    
 12  district               12464 non-null  str    
 13  city                   12464 non-null  str    
 14  geo_region             12464 non-null  str    
 15  living_ratio 

In [77]:
df_kyiv.info()

<class 'pandas.DataFrame'>
RangeIndex: 10958 entries, 0 to 10957
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   num_of_rooms       10958 non-null  Int64  
 1   freshly_renovated  10958 non-null  bool   
 2   area               10958 non-null  float64
 3   living_area        10958 non-null  float64
 4   kitchen_area       10958 non-null  float64
 5   floor              10958 non-null  Int64  
 6   floors_in_house    10958 non-null  Int64  
 7   year_of_building   10958 non-null  Int64  
 8   price              10958 non-null  float64
 9   house_type         10958 non-null  str    
 10  heating            10958 non-null  str    
 11  wall_type          10958 non-null  str    
 12  district           10958 non-null  str    
 13  city               10958 non-null  str    
 14  geo_region         10958 non-null  str    
 15  living_ratio       10958 non-null  float64
 16  is_low_floor       10958 non-null

In [78]:
df_ukraine = df_ukraine[df_ukraine["freshly_renovated"].notnull()]
df_kyiv = df_kyiv[df_kyiv["freshly_renovated"].notnull()]

In [79]:
cols_to_clean = ["price", "area"]

def remove_outliers_iqr(group):
    clean_group = group.copy()
    
    for col in cols_to_clean:
        Q1 = clean_group[col].quantile(0.25)
        Q3 = clean_group[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        clean_group = clean_group[(clean_group[col] >= lower_bound) & (clean_group[col] <= upper_bound)]
        
    return clean_group

df_ukraine_clean = remove_outliers_iqr(df_ukraine)
df_kyiv_clean = remove_outliers_iqr(df_kyiv)

In [80]:
df_ukraine_clean.describe()

,num_of_rooms,area,living_area,kitchen_area,floor,floors_in_house,year_of_building,price,living_ratio,is_large_city,is_close_to_eu,is_near_border_threat,is_tourist_hub,is_low_floor,is_high_floor,is_historic,is_soviet,is_modern
count,11345.0,11345.000000,11345.000000,11345.000000,11345.0,11345.0,11345.0,11345.000000,11345.000000,11345.000000,11345.000000,11345.000000,11345.000000,11345.000000,11345.000000,11345.000000,11345.000000,11345.000000
mean,1.950815,57.306764,31.596859,12.353312,6.065227,9.890172,1995.097312,1106.464571,0.558508,0.700573,0.273513,0.208374,0.495284,0.202556,0.337858,0.040194,0.435522,0.524284
std,0.852305,20.601713,13.226347,7.097009,4.485144,5.639211,30.176012,503.065215,0.149398,0.458027,0.445781,0.406163,0.500000,0.401922,0.473001,0.196422,0.495847,0.499432
min,1.0,8.600000,1.000000,1.000000,1.0,1.0,1507.0,24.000000,0.008621,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.0,43.000000,20.000000,7.000000,3.0,5.0,1976.0,734.000000,0.462886,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2.0,54.000000,30.000000,10.000000,5.0,9.0,2005.0,995.000000,0.557107,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,3.0,69.300000,40.000000,15.300000,9.0,12.0,2020.0,1400.000000,0.649486,1.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000
max,6.0,118.000000,100.000000,99.000000,28.0,36.0,2028.0,2622.000000,2.090318,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [81]:
df_ukraine[df_ukraine["price"] == 24]

,num_of_rooms,freshly_renovated,area,living_area,kitchen_area,floor,floors_in_house,year_of_building,price,house_type,...,living_ratio,is_large_city,is_close_to_eu,is_near_border_threat,is_tourist_hub,is_low_floor,is_high_floor,is_historic,is_soviet,is_modern
2497,1,False,43.9,12.2,20.3,4,6,2026,24.0,new_build,...,0.277904,0,1,0,1,0,0,0,0,1


In [82]:
df_ukraine_clean = df_ukraine_clean[df_ukraine_clean['kitchen_area'] < df_ukraine_clean['area']]
df_ukraine_clean = df_ukraine_clean[df_ukraine_clean['living_area'] < df_ukraine_clean['area']]
df_ukraine_clean = df_ukraine_clean[(df_ukraine_clean['living_area'] + df_ukraine_clean['kitchen_area']) <= df_ukraine_clean['area']]

df_ukraine_clean = df_ukraine_clean[df_ukraine_clean['floors_in_house'] <= 50]
df_ukraine_clean = df_ukraine_clean[df_ukraine_clean['floor'] <= df_ukraine_clean['floors_in_house']]

df_ukraine_clean = df_ukraine_clean[df_ukraine_clean['area'] >= 15]
df_ukraine_clean = df_ukraine_clean[df_ukraine_clean['living_area'] >= 9]
df_ukraine_clean = df_ukraine_clean[df_ukraine_clean['kitchen_area'] >= 4]
df_ukraine_clean = df_ukraine_clean[df_ukraine_clean['price'] >= 200]

df_ukraine_clean.describe()

,num_of_rooms,area,living_area,kitchen_area,floor,floors_in_house,year_of_building,price,living_ratio,is_large_city,is_close_to_eu,is_near_border_threat,is_tourist_hub,is_low_floor,is_high_floor,is_historic,is_soviet,is_modern
count,10750.0,10750.000000,10750.000000,10750.000000,10750.0,10750.0,10750.0,10750.000000,10750.000000,10750.000000,10750.000000,10750.000000,10750.000000,10750.000000,10750.000000,10750.000000,10750.000000,10750.000000
mean,1.981116,58.362806,31.763590,12.398437,6.12586,9.984837,1995.254698,1113.319516,0.545249,0.702140,0.281581,0.202140,0.508930,0.199907,0.338698,0.040186,0.432837,0.526977
std,0.85104,20.168913,13.150293,6.795965,4.497058,5.636775,30.156028,501.073479,0.126421,0.457339,0.449791,0.401614,0.499943,0.399949,0.473289,0.196404,0.495492,0.499295
min,1.0,15.000000,9.000000,4.000000,1.0,1.0,1507.0,210.000000,0.128205,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.0,43.500000,20.000000,7.000000,3.0,5.0,1976.0,742.000000,0.458333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2.0,55.000000,30.000000,10.000000,5.0,9.0,2006.0,1000.000000,0.550543,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,3.0,70.000000,40.000000,15.300000,9.0,12.0,2020.0,1403.750000,0.636364,1.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000
max,6.0,118.000000,100.000000,68.000000,28.0,36.0,2028.0,2622.000000,0.921522,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [83]:
df_ukraine_clean.info()

<class 'pandas.DataFrame'>
Index: 10750 entries, 0 to 12463
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   num_of_rooms           10750 non-null  Int64  
 1   freshly_renovated      10750 non-null  bool   
 2   area                   10750 non-null  float64
 3   living_area            10750 non-null  float64
 4   kitchen_area           10750 non-null  float64
 5   floor                  10750 non-null  Int64  
 6   floors_in_house        10750 non-null  Int64  
 7   year_of_building       10750 non-null  Int64  
 8   price                  10750 non-null  float64
 9   house_type             10750 non-null  str    
 10  heating                10750 non-null  str    
 11  wall_type              10750 non-null  str    
 12  district               10750 non-null  str    
 13  city                   10750 non-null  str    
 14  geo_region             10750 non-null  str    
 15  living_ratio      

In [84]:
df_kyiv_clean.info()

<class 'pandas.DataFrame'>
Index: 9811 entries, 0 to 10957
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   num_of_rooms       9811 non-null   Int64  
 1   freshly_renovated  9811 non-null   bool   
 2   area               9811 non-null   float64
 3   living_area        9811 non-null   float64
 4   kitchen_area       9811 non-null   float64
 5   floor              9811 non-null   Int64  
 6   floors_in_house    9811 non-null   Int64  
 7   year_of_building   9811 non-null   Int64  
 8   price              9811 non-null   float64
 9   house_type         9811 non-null   str    
 10  heating            9811 non-null   str    
 11  wall_type          9811 non-null   str    
 12  district           9811 non-null   str    
 13  city               9811 non-null   str    
 14  geo_region         9811 non-null   str    
 15  living_ratio       9811 non-null   float64
 16  is_low_floor       9811 non-null   int6

In [85]:
df_kyiv_clean = df_kyiv_clean[df_kyiv_clean['kitchen_area'] < df_kyiv_clean['area']]
df_kyiv_clean = df_kyiv_clean[df_kyiv_clean['living_area'] < df_kyiv_clean['area']]
df_kyiv_clean = df_kyiv_clean[(df_kyiv_clean['living_area'] + df_kyiv_clean['kitchen_area']) <= df_kyiv_clean['area']]

df_kyiv_clean = df_kyiv_clean[df_kyiv_clean['floors_in_house'] <= 50]
df_kyiv_clean = df_kyiv_clean[df_kyiv_clean['floor'] <= df_kyiv_clean['floors_in_house']]

df_kyiv_clean = df_kyiv_clean[df_kyiv_clean['area'] >= 15]
df_kyiv_clean = df_kyiv_clean[df_kyiv_clean['living_area'] >= 9]
df_kyiv_clean = df_kyiv_clean[df_kyiv_clean['kitchen_area'] >= 4]
df_kyiv_clean = df_kyiv_clean[df_kyiv_clean['price'] >= 200]

df_kyiv_clean.describe()

,num_of_rooms,area,living_area,kitchen_area,floor,floors_in_house,year_of_building,price,living_ratio,is_low_floor,is_high_floor,is_middle_floor,is_historic,is_soviet,is_modern
count,9425.0,9425.000000,9425.000000,9425.000000,9425.0,9425.0,9425.0,9425.000000,9425.000000,9425.000000,9425.000000,9425.000000,9425.000000,9425.000000,9425.000000
mean,2.090928,68.523639,34.798704,14.489491,9.843183,16.946737,1999.710875,1832.139874,0.509473,0.137613,0.242016,0.620477,0.035225,0.312573,0.652202
std,0.908545,28.571586,16.701560,8.105941,6.902595,8.501643,27.393479,723.594315,0.119564,0.344511,0.428327,0.485294,0.184359,0.463566,0.476297
min,1.0,17.400000,9.000000,4.000000,1.0,1.0,1858.0,440.000000,0.115237,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.0,45.900000,21.000000,8.000000,4.0,9.0,1980.0,1288.000000,0.422222,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2.0,63.000000,31.664031,12.500000,8.0,16.0,2012.0,1658.000000,0.510870,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000
75%,3.0,85.000000,43.454577,18.000000,15.0,25.0,2021.0,2248.000000,0.593220,0.000000,0.000000,1.000000,0.000000,1.000000,1.000000
max,6.0,156.300000,125.000000,73.900000,36.0,47.0,2028.0,4111.000000,0.945695,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [86]:
df_kyiv_clean.info()

<class 'pandas.DataFrame'>
Index: 9425 entries, 0 to 10957
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   num_of_rooms       9425 non-null   Int64  
 1   freshly_renovated  9425 non-null   bool   
 2   area               9425 non-null   float64
 3   living_area        9425 non-null   float64
 4   kitchen_area       9425 non-null   float64
 5   floor              9425 non-null   Int64  
 6   floors_in_house    9425 non-null   Int64  
 7   year_of_building   9425 non-null   Int64  
 8   price              9425 non-null   float64
 9   house_type         9425 non-null   str    
 10  heating            9425 non-null   str    
 11  wall_type          9425 non-null   str    
 12  district           9425 non-null   str    
 13  city               9425 non-null   str    
 14  geo_region         9425 non-null   str    
 15  living_ratio       9425 non-null   float64
 16  is_low_floor       9425 non-null   int6

In [87]:
df_kyiv_clean.drop(["city", "geo_region"], axis=1, inplace=True)

In [88]:
categorical_cols_kyiv = ['district', 'freshly_renovated'] 

df_kyiv_encoded = pd.get_dummies(df_kyiv_clean, columns=categorical_cols_kyiv, drop_first=True, dtype=int)

print("Колонки київської моделі після OHE:")
print(df_kyiv_encoded.columns.tolist())

Колонки київської моделі після OHE:
['num_of_rooms', 'area', 'living_area', 'kitchen_area', 'floor', 'floors_in_house', 'year_of_building', 'price', 'house_type', 'heating', 'wall_type', 'living_ratio', 'is_low_floor', 'is_high_floor', 'is_middle_floor', 'is_historic', 'is_soviet', 'is_modern', 'district_Desnianskyi', 'district_Dniprovskyi', 'district_Holosiivskyi', 'district_Obolonskyi', 'district_Pecherskyi', 'district_Podilskyi', 'district_Shevchenkivskyi', 'district_Solomianskyi', 'district_Sviatoshynskyi', 'freshly_renovated_True']


In [89]:
categorical_cols_regions = ['city', 'district', 'freshly_renovated', 'geo_region']

df_ukraine_encoded = pd.get_dummies(df_ukraine_clean, columns=categorical_cols_regions, drop_first=True, dtype=int)


print("\nКолонки моделі регіонів після OHE:")
print(df_ukraine_encoded.columns.tolist())


Колонки моделі регіонів після OHE:
['num_of_rooms', 'area', 'living_area', 'kitchen_area', 'floor', 'floors_in_house', 'year_of_building', 'price', 'house_type', 'heating', 'wall_type', 'living_ratio', 'is_large_city', 'is_close_to_eu', 'is_near_border_threat', 'is_tourist_hub', 'is_low_floor', 'is_high_floor', 'is_historic', 'is_soviet', 'is_modern', 'city_Chernihiv', 'city_Chernivtsi', 'city_Dnipro', 'city_Ivano-Frankivsk', 'city_Kharkiv', 'city_Kherson', 'city_Khmelnytskyi', 'city_Kropyvnytskyi', 'city_Lutsk', 'city_Lviv', 'city_Mykolaiv', 'city_Odesa', 'city_Poltava', 'city_Rivne', 'city_Sumy', 'city_Ternopil', 'city_Uzhhorod', 'city_Vinnytsia', 'city_Zaporizhzhia', 'city_Zhytomyr', 'district_Outskirts', 'district_Residential', 'freshly_renovated_True', 'geo_region_East', 'geo_region_North', 'geo_region_South', 'geo_region_West']


In [90]:
df_ukraine_encoded.to_csv("Ukraine.csv")
df_kyiv_encoded.to_csv("Kyiv_ML.csv")